In [ ]:
import os

NOTEBOOK_1_OUTPUT_NAME = "Livrable3_Features" 
# ---

# --- 2. Définition des Chemins (ne pas modifier) ---
# Chemins d'ENTRÉE (lecture depuis la sortie du notebook 1)
INPUT_DIR = "/kaggle/input/livrable3-features"
CAPTIONS_FILE = os.path.join(INPUT_DIR, 'captions.txt')
FEATURES_FILE = os.path.join(INPUT_DIR, 'features.pkl')

# Chemins de SORTIE (écriture dans le dossier local)
OUTPUT_DIR = "/kaggle/working/"
TOKENIZER_FILE = os.path.join(OUTPUT_DIR, 'tokenizer.pkl')
MODEL_FILE = os.path.join(OUTPUT_DIR, 'image_captioning_model.h5')
CURVES_FILE = os.path.join(OUTPUT_DIR, 'training_curves.png')
MODEL_PLOT_FILE = os.path.join(OUTPUT_DIR, 'model_rnn_decoder.png')

# --- 3. Vérification des Chemins ---
print("--- Vérification des chemins d'entrée ---")
if not os.path.exists(FEATURES_FILE):
    print(f"ERREUR : Le fichier 'features.pkl' est introuvable à cet emplacement : {FEATURES_FILE}")
    print("Veuillez vérifier que 'NOTEBOOK_1_OUTPUT_NAME' est correct.")
else:
    print(f"Fichier 'features.pkl' trouvé. Prêt à continuer.")

if not os.path.exists(CAPTIONS_FILE):
    print(f"ERREUR : Le fichier 'captions.txt' est introuvable à cet emplacement : {CAPTIONS_FILE}")
else:
    print(f"Fichier 'captions.txt' trouvé. Prêt à continuer.")

print(f"\nLes sorties seront sauvegardées dans : {OUTPUT_DIR}")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Embedding, LSTM, Dropout, Add
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical, plot_model

import numpy as np
import pickle
import string
import os
from tqdm import tqdm
import matplotlib.pyplot as plt

# Vérification GPU
print("--- Vérification du GPU ---")
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    print("ATTENTION : Aucun GPU n'est détecté. L'entraînement sera extrêmement lente.")
else:
    print(f"GPU détecté : {gpus[0].name}")

In [ ]:
# --- 1. Fonctions de Chargement et Préparation ---

def load_captions(filename):
    captions_dict = {}
    with open(filename, 'r') as f:
        for line in f:
            tokens = line.split()
            if len(line) < 2: continue
            image_id, image_caption = tokens[0], tokens[1:]
            image_caption = ' '.join(image_caption)
            if image_id not in captions_dict:
                captions_dict[image_id] = []
            captions_dict[image_id].append(image_caption)
    return captions_dict

def clean_captions(captions_dict):
    table = str.maketrans('', '', string.punctuation)
    for key, caption_list in captions_dict.items():
        for i in range(len(caption_list)):
            caption = caption_list[i]
            caption = caption.split()
            caption = [word.lower() for word in caption]
            caption = [w.translate(table) for w in caption]
            caption = [word for word in caption if len(word) > 1]
            caption = [word for word in caption if word.isalpha()]
            caption_list[i] = '<start> ' + ' '.join(caption) + ' <end>'
    return captions_dict

def load_image_features(filename):
    with open(filename, 'rb') as f:
        features = pickle.load(f)
    return features

def to_vocabulary(captions_dict):
    all_captions = set()
    for key in captions_dict.keys():
        [all_captions.add(d) for d in captions_dict[key]]
    return list(all_captions)

def create_tokenizer(captions_list):
    tokenizer = Tokenizer(num_words=10000, oov_token="<unk>")
    tokenizer.fit_on_texts(captions_list)
    vocab_size = len(tokenizer.word_index) + 1
    return tokenizer, vocab_size

def get_max_length(captions_dict):
    all_captions = to_vocabulary(captions_dict)
    return max(len(d.split()) for d in all_captions)

# --- 2. Générateur de Données (AVEC LE FIX TUPLE) ---

def data_generator(captions_dict, features_dict, tokenizer, max_length, vocab_size, batch_size):
    X1_batch, X2_batch, y_batch = [], [], []
    n = 0
    while True:
        for image_id, caption_list in captions_dict.items():
            if image_id not in features_dict:
                continue
            image_features = features_dict[image_id]
            for caption in caption_list:
                n += 1
                sequence = tokenizer.texts_to_sequences([caption])[0]
                for i in range(1, len(sequence)):
                    X1_batch.append(image_features)
                    in_seq = sequence[:i]
                    in_seq = pad_sequences([in_seq], maxlen=max_length, padding='post')[0]
                    X2_batch.append(in_seq)
                    out_word = sequence[i]
                    out_word_one_hot = to_categorical([out_word], num_classes=vocab_size)[0]
                    y_batch.append(out_word_one_hot)
                if n == batch_size:
                    # FIX : Yield des inputs sous forme de TUPLE, pas de liste
                    yield ((np.array(X1_batch), np.array(X2_batch)), np.array(y_batch))
                    X1_batch, X2_batch, y_batch = [], [], []
                    n = 0

# --- 3. Définition du Modèle RNN (Décodeur) ---

def build_rnn_decoder(vocab_size, max_length, embedding_dim, lstm_units, feature_dim):
    input_features = Input(shape=(feature_dim,))
    fe1 = Dropout(0.4)(input_features)
    fe2 = Dense(256, activation='relu')(fe1)

    input_sequence = Input(shape=(max_length,))
    se1 = Embedding(vocab_size, embedding_dim, mask_zero=True)(input_sequence)
    se2 = Dropout(0.4)(se1)
    se3 = LSTM(256)(se2)

    decoder1 = Add()([fe2, se3])
    decoder2 = Dense(256, activation='relu')(decoder1)
    outputs = Dense(vocab_size, activation='softmax')(decoder2)
    
    model = Model(inputs=[input_features, input_sequence], outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# --- 4. Script d'Exécution Principal (Training) ---

# --- Paramètres ---
EPOCHS = 5
BATCH_SIZE = 64
EMBEDDING_DIM = 256
LSTM_UNITS = 256
FEATURE_DIM = 2048 # InceptionV3

print("--- 1. Chargement des données (captions.txt) ---")
captions_dict = load_captions(CAPTIONS_FILE)
print(f"Légendes chargées : {len(captions_dict)} images.")

print("--- 2. Nettoyage des légendes ---")
captions_dict = clean_captions(captions_dict)

print("--- 3. Chargement des features (features.pkl) ---")
features_dict = load_image_features(FEATURES_FILE)
print(f"Features chargées : {len(features_dict)} images.")

print("--- 4. Synchronisation données ---")
captions_train = {}
for img_id, caps in captions_dict.items():
    if img_id in features_dict:
        captions_train[img_id] = caps
print(f"Données d'entraînement finales : {len(captions_train)} images.")

print("--- 5. Création du Tokenizer ---")
all_captions_list = to_vocabulary(captions_train)
tokenizer, vocab_size = create_tokenizer(all_captions_list)
max_length = get_max_length(captions_train)
print(f"Taille du vocabulaire : {vocab_size}")
print(f"Longueur max des légendes : {max_length}")

print(f"Sauvegarde du tokenizer dans {TOKENIZER_FILE}...")
with open(TOKENIZER_FILE, 'wb') as f:
    pickle.dump(tokenizer, f)

print("--- 6. Création du Modèle RNN ---")
model = build_rnn_decoder(vocab_size, max_length, EMBEDDING_DIM, LSTM_UNITS, FEATURE_DIM)
model.summary()
plot_model(model, to_file=MODEL_PLOT_FILE, show_shapes=True)

print("--- 7. Lancement de l'Entraînement ---")
total_sequences = 0
for cap_list in captions_train.values():
    for cap in cap_list:
        total_sequences += len(cap.split()) - 1
steps_per_epoch = total_sequences // BATCH_SIZE
if total_sequences % BATCH_SIZE != 0: steps_per_epoch += 1

print(f"Total Séquences: {total_sequences}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Steps per Epoch: {steps_per_epoch}")

# --- FIX : Création du tf.data.Dataset ---
# 1. Définir la signature de sortie (correspond à notre yield)
output_signature = (
    (tf.TensorSpec(shape=(None, FEATURE_DIM), dtype=tf.float32),  # X1: features
     tf.TensorSpec(shape=(None, max_length), dtype=tf.int32)),  # X2: sequence
    tf.TensorSpec(shape=(None, vocab_size), dtype=tf.float32)    # y: one-hot word
)

# 2. Créer l'objet tf.data.Dataset
train_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(captions_train, features_dict, tokenizer, max_length, vocab_size, BATCH_SIZE),
    output_signature=output_signature
)

# 3. Optimiser les performances
train_dataset = train_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

# 4. Appeler model.fit() avec le nouveau Dataset
history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    verbose=1
)

print(f"--- 8. Entraînement Terminé. Sauvegarde du modèle dans {MODEL_FILE} ---")
model.save(MODEL_FILE)

print("--- 9. Sauvegarde des courbes d'entraînement ---")
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Perte')
plt.title('Évolution de la Perte')
plt.xlabel('Époque')
plt.ylabel('Loss')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Précision')
plt.title('Évolution de la Précision')
plt.xlabel('Époque')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.savefig(CURVES_FILE)

print(f"\n--- PROJET TERMINÉ. Vos fichiers sont dans {OUTPUT_DIR} ---")